# Part 18: Fine-tuning & Dataset Curation

> Building custom training datasets, curating data quality, and fine-tuning LLMs for domain-specific tasks — with the product pricing use case.

---


## 18.1 When to Fine-tune?

| Approach | When to Use | Cost |
|----------|------------|------|
| Prompting | General tasks, fast iteration | Cheapest |
| RAG | Private data, recent info | Medium |
| Fine-tuning | Consistent style, domain expertise, latency | Higher |
| Train from scratch | Unique domain, no existing model | Very High |

**Fine-tuning is best when:**
- You have 100s–1000s of labeled examples
- You need consistent tone/format
- The task is highly specific
- Inference latency matters


## 18.2 Loading Training Data from HuggingFace

In [ ]:
from datasets import load_dataset
import pandas as pd

# Load Amazon product reviews dataset
dataset = load_dataset("McAuley-Lab/Amazon-Reviews-2023", 
                       "raw_meta_Appliances",
                       split="full",
                       trust_remote_code=True)

# Convert to DataFrame for easier manipulation
df = pd.DataFrame(dataset)
print(f"Total records: {len(df)}")
print(df.columns.tolist())
print(df.head(2))


## 18.3 Data Filtering & Quality Assessment

In [ ]:
# Filter: only products with price in target range
def filter_products(df: pd.DataFrame, min_price: float = 1.0, max_price: float = 999.0) -> pd.DataFrame:
    """Keep only products with valid prices in range."""
    df = df.dropna(subset=["price"])
    df = df[df["price"].apply(lambda x: isinstance(x, (int, float)))]
    df = df[(df["price"] >= min_price) & (df["price"] <= max_price)]
    return df

# Analyze price distribution
import matplotlib.pyplot as plt

def analyze_prices(df: pd.DataFrame):
    prices = df["price"]
    print(f"Count:  {len(prices)}")
    print(f"Mean:   ${prices.mean():.2f}")
    print(f"Median: ${prices.median():.2f}")
    print(f"Std:    ${prices.std():.2f}")
    
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.hist(prices, bins=50, edgecolor='black')
    plt.title("Price Distribution")
    plt.xlabel("Price ($)")
    
    plt.subplot(1, 2, 2)
    plt.hist(prices[prices < 100], bins=50, edgecolor='black')
    plt.title("Price Distribution (< $100)")
    plt.xlabel("Price ($)")
    plt.tight_layout()
    plt.show()


## 18.4 Item Class — Data Normalization

In [ ]:
from dataclasses import dataclass, field
from typing import Optional
import re

@dataclass
class Item:
    """Normalized product item for training data."""
    title: str
    price: float
    category: str
    description: str = ""
    features: list[str] = field(default_factory=list)
    
    def clean_text(self, text: str) -> str:
        """Remove HTML, extra whitespace, special chars."""
        text = re.sub(r'<[^>]+>', '', text)      # remove HTML
        text = re.sub(r'[^\w\s.,!?-]', ' ', text) # keep basic punctuation
        text = re.sub(r'\s+', ' ', text)           # normalize whitespace
        return text.strip()
    
    def to_prompt(self) -> str:
        """Create input prompt for training."""
        parts = [f"Title: {self.title}"]
        if self.description:
            parts.append(f"Description: {self.clean_text(self.description)[:200]}")
        if self.features:
            parts.append(f"Features: {'; '.join(self.features[:3])}")
        parts.append(f"Category: {self.category}")
        return "\n".join(parts)
    
    def to_price_label(self) -> str:
        """Format price as training label."""
        return f"Price: ${self.price:.2f}"
    
    def to_training_example(self) -> dict:
        """Format as OpenAI fine-tuning JSONL format."""
        return {
            "messages": [
                {"role": "system",    "content": "Estimate the price of this product."},
                {"role": "user",      "content": self.to_prompt()},
                {"role": "assistant", "content": self.to_price_label()}
            ]
        }


## 18.5 Token Analysis

In [ ]:
import tiktoken

def count_tokens(text: str, model: str = "gpt-4o-mini") -> int:
    enc = tiktoken.encoding_for_model(model)
    return len(enc.encode(text))

def analyze_token_distribution(items: list[Item]):
    """Check if training examples fit within context limits."""
    token_counts = [count_tokens(item.to_prompt()) for item in items]
    
    print(f"Token stats:")
    print(f"  Mean:    {sum(token_counts)/len(token_counts):.0f}")
    print(f"  Max:     {max(token_counts)}")
    print(f"  >512 tk: {sum(1 for t in token_counts if t > 512)} ({sum(1 for t in token_counts if t > 512)/len(token_counts)*100:.1f}%)")
    
    # For fine-tuning, keep examples under 512 tokens
    valid = [(item, t) for item, t in zip(items, token_counts) if t <= 512]
    print(f"  Valid:   {len(valid)}/{len(items)}")
    return [item for item, _ in valid]


## 18.6 Train/Test Split & JSONL Export

In [ ]:
import json, random

def create_training_dataset(items: list[Item], train_ratio: float = 0.9) -> tuple:
    """Split items into train/test and export as JSONL."""
    random.shuffle(items)
    split = int(len(items) * train_ratio)
    train_items, test_items = items[:split], items[split:]
    
    def write_jsonl(items: list[Item], path: str):
        with open(path, "w") as f:
            for item in items:
                f.write(json.dumps(item.to_training_example()) + "\n")
        print(f"Wrote {len(items)} examples to {path}")
    
    write_jsonl(train_items, "train.jsonl")
    write_jsonl(test_items,  "test.jsonl")
    return train_items, test_items

# Example JSONL format (OpenAI fine-tuning)
example = {
    "messages": [
        {"role": "system",    "content": "Estimate the price of this product."},
        {"role": "user",      "content": "Title: Stainless Steel Coffee Maker\nCategory: Appliances"},
        {"role": "assistant", "content": "Price: $89.99"}
    ]
}
print(json.dumps(example, indent=2))


## 18.7 Fine-tuning via OpenAI API

In [ ]:
from openai import OpenAI
client = OpenAI()

# Step 1: Upload training file
def upload_training_file(file_path: str) -> str:
    with open(file_path, "rb") as f:
        response = client.files.create(file=f, purpose="fine-tune")
    print(f"Uploaded: {response.id}")
    return response.id

# Step 2: Create fine-tuning job
def start_fine_tuning(train_file_id: str, validation_file_id: str = None) -> str:
    job = client.fine_tuning.jobs.create(
        training_file=train_file_id,
        validation_file=validation_file_id,
        model="gpt-4o-mini-2024-07-18",
        hyperparameters={
            "n_epochs": 3,
            "learning_rate_multiplier": 1.8
        }
    )
    print(f"Fine-tuning job: {job.id}")
    return job.id

# Step 3: Monitor progress
def monitor_job(job_id: str):
    job = client.fine_tuning.jobs.retrieve(job_id)
    print(f"Status:     {job.status}")
    print(f"Model:      {job.fine_tuned_model}")
    print(f"Created at: {job.created_at}")
    
    # List events
    events = client.fine_tuning.jobs.list_events(job_id, limit=10)
    for event in events.data:
        print(f"  [{event.created_at}] {event.message}")


## 18.8 Evaluating Fine-tuned Models

In [ ]:
def evaluate_pricing_model(model_id: str, test_items: list[Item]) -> dict:
    """Compare fine-tuned model predictions vs actual prices."""
    results = []
    for item in test_items[:50]:  # sample
        response = client.chat.completions.create(
            model=model_id,
            messages=[
                {"role": "system", "content": "Estimate the price of this product."},
                {"role": "user",   "content": item.to_prompt()}
            ]
        )
        predicted_text = response.choices[0].message.content
        
        # Parse predicted price
        import re
        match = re.search(r'\$([\d.]+)', predicted_text)
        predicted_price = float(match.group(1)) if match else 0
        
        error_pct = abs(predicted_price - item.price) / item.price * 100
        results.append({
            "actual": item.price,
            "predicted": predicted_price,
            "error_pct": error_pct
        })
    
    avg_error = sum(r["error_pct"] for r in results) / len(results)
    good_estimates = sum(1 for r in results if r["error_pct"] < 20)
    
    return {
        "avg_error_pct": avg_error,
        "good_estimates_pct": good_estimates / len(results) * 100,
        "n_samples": len(results)
    }


## 18.9 Fine-tuning with HuggingFace (LoRA)

In [ ]:
# For open-source models — see Part10 for LoRA/QLoRA theory
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM

def setup_lora_training(model_name: str = "microsoft/phi-2"):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")
    
    lora_config = LoraConfig(
        r=16,               # rank
        lora_alpha=32,      # scaling
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    return model, tokenizer

training_args = TrainingArguments(
    output_dir="./fine_tuned_model",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    evaluation_strategy="epoch"
)


## 18.10 Summary

| Step | Action |
|------|--------|
| 1. Load | HuggingFace datasets, CSV, JSON |
| 2. Filter | Price range, text quality, duplicates |
| 3. Normalize | Item class, clean text |
| 4. Analyze | Token counts, distributions |
| 5. Split | 90% train / 10% test |
| 6. Export | JSONL format for OpenAI |
| 7. Upload | client.files.create() |
| 8. Train | client.fine_tuning.jobs.create() |
| 9. Evaluate | Price error %, good estimates % |

---

**Next:** [Part 19 — Production Deployment with Modal](Part19_Production_Deployment_Modal.ipynb)
